# 🏥 ExplainableVLM-Rad: Google Colab Paper Reproduction Launcher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikhram-S/EXPLAINABLE-VLM-PAPER/blob/main/notebooks/01_colab_kaggle_launcher.ipynb)

This notebook provides the complete, end-to-end paper reproduction workflow for **ExplainableVLM-Rad**: An Explainable Vision-Language Model for Automated Chest Radiography Report Generation with Supervised Visual Phrase-Grounding.

---

### ⚡ T4 GPU Performance Optimizations (Aug 2025)
This notebook includes the following T4-specific speed-ups:
- **KV-cached autoregressive generation** — primes the 196-token visual prefix once, then decodes with cached keys/values (5–10× faster validation)
- **`val_max_batches: 20`** — caps expensive generation to 20 batches per epoch (~80% less val time)
- **`batch_size: 8` + `grad_accum: 2`** — same effective batch of 16, but lighter VRAM per step
- **`subsample_size: 5000`** — ~3,500 train / 500 val / 1,000 test samples for T4-friendly epochs
- **`cudnn.benchmark = True`** + **`prefetch_factor=2`** — overlap data loading with GPU compute

> **Expected epoch time on T4**: ~2–4 minutes (previously 20+ min)

## 1. Environment Setup & Package Installation

In [ ]:
import os
import sys

# Ensure base directory is /content in Colab
if 'google.colab' in sys.modules:
    os.chdir('/content')

REPO_NAME = 'EXPLAINABLE-VLM-PAPER'
REPO_URL = 'https://github.com/Vikhram-S/EXPLAINABLE-VLM-PAPER.git'

if os.path.exists(REPO_NAME):
    print(f"🔄 Updating existing {REPO_NAME} repository...")
    os.chdir(REPO_NAME)
    !git pull origin main
else:
    print(f"📥 Cloning {REPO_NAME} repository...")
    !git clone {REPO_URL}
    os.chdir(REPO_NAME)

# Install dependencies & editable package
!pip install -q -r requirements.txt
!pip install -q -e .

print(f"✅ Active Working Directory: {os.getcwd()}")
print("✅ Package explainablevlm-rad installed successfully!")

## 2. Mount Google Drive & Verify GPU Hardware

In [ ]:
import sys
import torch

# Mount Google Drive if in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PERSISTENT_DIR = '/content/drive/MyDrive/explainablevlm_rad_outputs'
else:
    PERSISTENT_DIR = 'outputs'

os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"📁 Output Directory: {PERSISTENT_DIR}")

# Verify GPU Hardware
print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"⚡ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU Model: {gpu_name}")
    print(f"🧠 VRAM: {vram_gb:.2f} GB")
    if 'T4' in gpu_name:
        print("✅ T4 GPU detected — optimized batch_size=8 config will be used (stage2_mimic_cxr.yaml)")
    elif vram_gb < 12:
        print("⚠️  Low VRAM GPU detected. If OOM occurs, reduce batch_size to 4 in configs/experiments/stage2_mimic_cxr.yaml")
else:
    print("⚠️  No GPU detected — training will be very slow on CPU!")
    print("   Go to Runtime → Change runtime type → GPU to enable T4.")

## 3. Data Pipeline Sanity Check

In [ ]:
# Verify dataset loader, patch grounding masks, and text preprocessing
!python scripts/sanity_check_data.py --num_samples 4

## 4. Execute Training Pipeline

- **Stage 1 (IU X-Ray Pre-training)**
- **Stage 2 (MIMIC-CXR Fine-tuning)** — T4-optimized: `batch_size=8`, `subsample=5000`, `val_max_batches=20`, KV-cached generation
- **Ablation Model (No Visual Grounding Loss)**

> 💡 **T4 tip**: Each epoch should take ~2–4 min. If you see OOM errors, restart runtime and reduce `batch_size` to `4` in `configs/experiments/stage2_mimic_cxr.yaml`.

In [ ]:
import time

print("🚀 Stage 1: IU X-Ray Pre-training")
t0 = time.time()
!python src/train.py --config configs/experiments/stage1_iu_xray.yaml
print(f"✅ Stage 1 done in {(time.time()-t0)/60:.1f} min")

print("\n🚀 Stage 2: MIMIC-CXR Fine-tuning (T4-optimized)")
t1 = time.time()
!python src/train.py --config configs/experiments/stage2_mimic_cxr.yaml
print(f"✅ Stage 2 done in {(time.time()-t1)/60:.1f} min")

print("\n🚀 Ablation Study: No Grounding Loss")
t2 = time.time()
!python src/train.py --config configs/experiments/ablation_no_exp_loss.yaml
print(f"✅ Ablation done in {(time.time()-t2)/60:.1f} min")

## 5. Master Evaluation Suite & Metrics Benchmark

Evaluates NLG (BLEU-1 to 4, ROUGE-L, CIDEr), Clinical F1 (CheXbert-14), and Visual Grounding IoU.

In [ ]:
!python src/eval/evaluator.py

## 6. Generate All 8 Journal-Quality Figures (300 DPI) & LaTeX Tables

In [ ]:
# Generate paper figures
!python src/viz/plot_curves.py
!python src/viz/plot_comparison.py
!python src/viz/plot_heatmaps.py
!python src/viz/plot_ablation.py
!python src/viz/plot_pathology.py
!python src/viz/plot_human_eval.py
!python src/viz/plot_faithfulness.py
!python src/viz/plot_qualitative_grid.py

# Generate camera-ready LaTeX tables
!python src/viz/generate_latex_tables.py

print("📊 All figures and LaTeX tables successfully generated in outputs/")

## 7. Manuscript Verification & Human Eval Sheet Generation

In [ ]:
# Text-data drift check between outputs/results_summary.json and manuscript draft
!python scripts/check_manuscript_consistency.py

# Generate randomized blinded evaluation sheet for board-certified radiologist scoring
!python scripts/generate_human_eval_sheet.py

## 8. Save Outputs to Google Drive (Persistent Backup)

In [ ]:
import shutil, os

if 'google.colab' in sys.modules and os.path.exists('/content/drive'):
    dest = '/content/drive/MyDrive/explainablevlm_rad_outputs'
    os.makedirs(dest, exist_ok=True)
    shutil.copytree('outputs', dest, dirs_exist_ok=True)
    print(f"✅ All outputs backed up to Google Drive: {dest}")
else:
    print("📂 Outputs are in the local ./outputs/ directory.")
    !ls -lh outputs/